# Training BLISSNet

In [1]:
from blissnet.blissnet import BLISSNet
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import xarray
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm
from dataclasses import dataclass
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


## Dataset Initialization

In [2]:
south_asia_ds = xarray.open_dataset('./datasets/south_asia_t2m.nc')
punjab_ds = xarray.open_dataset('./datasets/punjab_t2m.nc')
karnatka_ds = xarray.open_dataset('./datasets/karnatka_t2m.nc')

print("Dataset XArray Loaded.")

Dataset XArray Loaded.


In [3]:
class T2MDataset(Dataset):
    def __init__(self, path, transform=None):
        super().__init__()
        xarray_ds = xarray.open_dataset(path)

        self.lat_max = xarray_ds.latitude.max().item()
        self.lat_min = xarray_ds.latitude.min().item()
        self.long_max = xarray_ds.longitude.max().item()
        self.long_min = xarray_ds.longitude.min().item()
        self.time_min = xarray_ds.time.min().item()
        self.time_max = xarray_ds.time.max().item()
        self.t2m_min = xarray_ds.t2m.min().item()
        self.t2m_max = xarray_ds.t2m.max().item()

        self.time_res = abs(xarray_ds.time[1] - xarray_ds.time[0]).item()
        self.lat_res = abs(xarray_ds.latitude[1] - xarray_ds.latitude[0]).item()
        self.long_res = abs(xarray_ds.longitude[1] - xarray_ds.longitude[0]).item()
        
        ds_t2m = torch.from_numpy(xarray_ds['t2m'].to_numpy())
        T, H, W = ds_t2m.shape
        ds_t2m = list(ds_t2m.reshape(T, -1).unsqueeze(dim=-1).unbind(dim=0))

        vec_lat = torch.from_numpy(xarray_ds.latitude.to_numpy())
        vec_long = torch.from_numpy(xarray_ds.longitude.to_numpy())
        arr_lat, arr_long = torch.meshgrid([vec_lat, vec_long], indexing='ij')
        grid_coord = list(torch.stack([arr_lat, arr_long], dim=-1).reshape(H*W,2).unsqueeze(0).expand(T, H*W, 2).unbind(dim=0))

        self.H = H
        self.W = W
        self.T = T
        self.dataset = list(zip(ds_t2m, grid_coord))
        self.transform = transform(
            self.t2m_min,
            self.t2m_max,
            self.lat_min,
            self.lat_max,
            self.long_min,
            self.long_max
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        if index >= len(self.dataset) or index < 0:
            raise IndexError

        item = self.dataset[index]
        if self.transform is not None:
            item = self.transform(item)

        return item

In [4]:
class T2MNormalize:
    def __init__(self, t2m_min, t2m_max, lat_min, lat_max, long_min, long_max):
        self.t2m_min = t2m_min
        self.t2m_max = t2m_max
        self.coord_min = torch.tensor([lat_min, long_min], dtype=torch.float32)
        self.coord_max = torch.tensor([lat_max, long_max], dtype=torch.float32)

    def __call__(self, sample):
        t2m, coord = sample

        t2m = (t2m - self.t2m_min) / (self.t2m_max - self.t2m_min)
        coord = (coord - self.coord_min) / (self.coord_max - self.coord_min)

        return t2m, coord

    def denormalize(self, sample):
        t2m, coord = sample

        t2m = t2m * (self.t2m_max - self.t2m_min) + self.t2m_min
        coord = coord * (self.coord_max - self.coord_min) + self.coord_min

        return t2m, coord

In [5]:
south_asia_dataset = T2MDataset('./datasets/south_asia_t2m.nc', transform=T2MNormalize)
punjab_dataset = T2MDataset('./datasets/punjab_t2m.nc', transform=T2MNormalize)
karnatka_dataset = T2MDataset('./datasets/karnatka_t2m.nc', transform=T2MNormalize)

## Training BLISSNet

In [6]:
class BLISSNetTrainer():
    def __init__(self, dataset:T2MDataset, config, device=None):
        self.config = config
        self.device = device
        if self.device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        dataset_size = len(dataset)
        train_size = int(0.9 * dataset_size)
        test_size = dataset_size - train_size

        trainset, testset = random_split(dataset, [train_size, test_size])
        self.trainloader = DataLoader(trainset, batch_size=config.batch_size, shuffle=True)
        self.testloader = DataLoader(testset, batch_size=config.batch_size, shuffle=False)

        self.transform = dataset.transform
        self.dataset_params = {'H':dataset.H, 'W':dataset.W, 'T':dataset.T,'lat_min':dataset.lat_min, 'lat_max':dataset.lat_max, 'long_min':dataset.long_min, 'long_max':dataset.long_max, 'lat_res':dataset.lat_res, 'long_res':dataset.long_res, 'time_res':dataset.time_res, 'time_min':dataset.time_min, 'time_max':dataset.time_max, 't2m_min':dataset.t2m_min, 't2m_max':dataset.t2m_max}

        self.model = BLISSNet(config.emb_dim, config.K, config.n_heads, config.dropout).to(self.device)
        
        self.epochs = config.epochs
        self.lr = config.lr
        config.grid_size = self.dataset_params['H'] * self.dataset_params['W']

        self.TRAIN_LOSSES_S1 = []
        self.TEST_LOSSES_S1 = []

        self.TRAIN_LOSSES_S2 = []
        self.TEST_LOSSES_S2 = []

    def pad_to_multiple(self, x, multiple=8):
        H, W = x.shape[-2:]
        pad_h = (multiple - H % multiple) % multiple
        pad_w = (multiple - W % multiple) % multiple
        x_padded = F.pad(x, (0, pad_w, 0, pad_h), mode='reflect')
        return x_padded, (H, W)

    def train_stage1(self):
        criterion = nn.MSELoss()
        self.model.train(True)
        running_train_loss = 0
        running_test_loss = 0
        total = 0
        for t2m, _ in tqdm(self.trainloader):
            t2m = t2m.to(self.device) # shape: B, N, 1
            B, N, _ = t2m.shape
            H, W = self.dataset_params['H'], self.dataset_params['W']
            t2m = t2m.permute(0, 2, 1).reshape(B, 1, H, W)

            t2m_padded, orig_size = self.pad_to_multiple(t2m)
            output, _, _ = self.model(t2m_padded, t2m_padded.shape[-2:], phase=0)
            output = output[..., :orig_size[0], :orig_size[1]]
            self.optimizer.zero_grad()
            loss = criterion(output, t2m)
            loss.backward()
            self.optimizer.step()

            total += B
            running_train_loss += loss.item() * B
        avg_train_loss = running_train_loss / total

        with torch.no_grad():
            self.model.eval()
            total = 0
            for t2m, _ in tqdm(self.testloader):
                t2m = t2m.to(self.device) # shape: B, N, 1
                B, N, _ = t2m.shape
                H, W = self.dataset_params['H'], self.dataset_params['W']
                t2m = t2m.permute(0, 2, 1).reshape(B, 1, H, W)

                t2m_padded, orig_size = self.pad_to_multiple(t2m)
                output, _, _ = self.model(t2m_padded, t2m_padded.shape[-2:], phase=0)
                output = output[..., :orig_size[0], :orig_size[1]]
                loss = criterion(output, t2m)
            
                total += B
                running_test_loss += loss.item() * B

        
        avg_test_loss = running_test_loss / total

        return avg_train_loss, avg_test_loss

    def generate_subsets(self, t2m_batch, coord_batch):
        B, N, _ = t2m_batch.shape
        min_size = max(1, int(0.005 * N))
        max_size = max(min_size + 1, int(0.05 * N))
        size = int(torch.randint(min_size, max_size, (1,)).item())

        indices = torch.rand(B, N, device=t2m_batch.device).argsort(dim=1)[:, :size]

        t2m_indices = indices.unsqueeze(-1)
        coord_indices = indices.unsqueeze(-1).expand(-1, -1, 2)

        subset_t2m = torch.gather(t2m_batch, dim=1, index=t2m_indices)
        subset_coord = torch.gather(coord_batch, dim=1, index=coord_indices)
        return subset_t2m, subset_coord, indices

    def lossEP(self, input_subset, output_recon, indices, lambda_ep):
        B, _, H, W = output_recon.shape
        output_recon = output_recon.reshape(B, 1, H*W).permute(0, 2, 1)
        t2m_indices = indices.unsqueeze(-1)
        output_subset = torch.gather(output_recon, dim=1, index=t2m_indices) # B, subset_size, 1
        mse = nn.MSELoss()
        return lambda_ep * mse(output_subset, input_subset)

    def lossEMB(self, input_emb, target_emb, lambda_emb):
        mse = nn.MSELoss()
        return lambda_emb * mse(target_emb, input_emb)

    def lossCoeff(self, input_coeff, target_coeff, lambda_coeff):
        mse = nn.MSELoss()
        return lambda_coeff * mse(target_coeff, input_coeff)

    def lossGT(self, input_t2m, target_t2m, lambda_gt):
        mse = nn.MSELoss(reduction='none')
        diff_sq = mse(target_t2m, input_t2m)

        l2_diff = torch.sqrt(torch.sum(diff_sq, dim=(1,2,3)))
        l2_true = torch.sqrt(torch.sum(input_t2m ** 2, dim=(1,2,3)))

        l_gt = torch.mean(l2_diff / (l2_true + 1e-8))
        return lambda_gt * l_gt

    def train_stage2(self, lambda_ep, lambda_emb, lambda_coeff, lambda_gt):
        self.model.train(True)
        self.model.branch1.eval()
        self.model.trunk_net.eval()
        running_train_loss = 0
        running_test_loss = 0
        total = 0

        for t2m, coord in tqdm(self.trainloader):
            t2m, coord = t2m.to(self.device), coord.to(self.device)
            B, N, _ = t2m.shape
            H, W = self.dataset_params['H'], self.dataset_params['W']
            t2m_grid = t2m.permute(0, 2, 1).reshape(B, 1, H, W)

            subset_t2m, subset_coord, subset_indices = self.generate_subsets(t2m, coord)

            output, coeff_stage2, emb_stage2 = self.model((subset_t2m, subset_coord), (H, W), phase=1)
            _, coeff_stage1, emb_stage1 = self.model(t2m_grid, (H, W), phase=0)

            loss_ep = self.lossEP(subset_t2m, output, subset_indices, lambda_ep)
            loss_emb = self.lossEMB(emb_stage2, emb_stage1, lambda_emb)
            loss_coeff = self.lossCoeff(coeff_stage2, coeff_stage1, lambda_coeff)
            loss_gt = self.lossGT(t2m_grid, output, lambda_gt)

            self.optimizer.zero_grad()
            loss = loss_ep + loss_emb + loss_coeff + loss_gt
            loss.backward()
            self.optimizer.step()

            running_train_loss += loss.item() * B
            total += B

        avg_train_loss = running_train_loss / total

        with torch.no_grad():
            self.model.eval()
            total = 0
            for t2m, coord in tqdm(self.testloader):
                t2m, coord = t2m.to(self.device), coord.to(self.device)
                B, N, _ = t2m.shape
                H, W = self.dataset_params['H'], self.dataset_params['W']
                t2m_grid = t2m.permute(0, 2, 1).reshape(B, 1, H, W)
    
                subset_t2m, subset_coord, subset_indices = self.generate_subsets(t2m, coord)
    
                output, coeff_stage2, emb_stage2 = self.model((subset_t2m, subset_coord), (H, W), phase=1)
                _, coeff_stage1, emb_stage1 = self.model(t2m_grid, (H, W), phase=0)
    
                loss_ep = self.lossEP(subset_t2m, output, subset_indices, lambda_ep)
                loss_emb = self.lossEMB(emb_stage2, emb_stage1, lambda_emb)
                loss_coeff = self.lossCoeff(coeff_stage2, coeff_stage1, lambda_coeff)
                loss_gt = self.lossGT(t2m_grid, output, lambda_gt)

                loss = loss_ep + loss_emb + loss_coeff + loss_gt
        
                running_test_loss += loss.item() * B
                total += B

        avg_test_loss = running_test_loss / total
        return avg_train_loss, avg_test_loss
            
    def train(self, lambda_ep, lambda_emb, lambda_coeff, lambda_gt):
        # stage 1
        print("========== Training Stage 1 ============")
    
        self.model.setPhase(0, self.config)
        self.model.to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), self.lr)    
        
        for i in range(self.epochs):
            train_loss, test_loss = self.train_stage1()
            self.TRAIN_LOSSES_S1.append(train_loss)
            self.TEST_LOSSES_S1.append(test_loss)

            print(f"Epoch {i + 1} | Train Loss: {train_loss} | Test Loss: {test_loss}")

        # stage 2
        print("========= Training Stage 2 ============")

        self.model.setPhase(1, self.config)
        self.model.to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), self.lr)  
        
        for i in range(self.epochs):
            train_loss, test_loss = self.train_stage2(lambda_ep, lambda_emb, lambda_coeff, lambda_gt)
            self.TRAIN_LOSSES_S2.append(train_loss)
            self.TEST_LOSSES_S2.append(test_loss)

            print(f"Epoch {i + 1} | Train Loss: {train_loss} | Test Loss: {test_loss}")

    @torch.no_grad()
    def super_resolve(self, multiplier, dataset:T2MDataset, timestep):
        self.model.eval()

        t2m, coords = dataset[timestep]
        H, W = dataset.H, dataset.W

        new_H = H * multiplier
        new_W = W * multiplier

        t2m = t2m.unsqueeze(0).to(self.device)  # (1, H*W, 1)

        coords = coords.unsqueeze(0)
        coords = coords.to(self.device)         # (1, H*W, 2)

        lat = torch.linspace(
            coords[..., 0].min(),
            coords[..., 0].max(),
            new_H,
            device=self.device,
        )

        lon = torch.linspace(
            coords[..., 1].min(),
            coords[..., 1].max(),
            new_W,
            device=self.device,
        )

        lat_grid, lon_grid = torch.meshgrid(lat, lon, indexing="ij")
        new_coords = torch.stack((lat_grid, lon_grid), dim=-1).reshape(
            1, new_H * new_W, 2
        )

        new_t2m, _, _ = self.model((t2m, coords), (new_H, new_W), phase=1)
        new_t2m = new_t2m.reshape(new_H, new_W)

        return new_t2m, new_coords


    def load(self, name):
        if not os.path.exists(f"./models/{name}_blissnet_chkpoint.pth"):
            return False

        with open(f"./models/{name}_blissnet_hist.pkl", 'rb') as file:
            data = pickle.load(file)
            self.TRAIN_LOSSES_S1 = data['train_loss_s1']
            self.TEST_LOSSES_S1 = data['test_loss_s1']
            self.TRAIN_LOSSES_S2 = data['train_loss_s2']
            self.TEST_LOSSES_S2 = data['test_loss_s2']

        self.model.setPhase(0, self.config)
        self.model.setPhase(1, self.config)
        self.model.to(self.device)

        chkpoint = torch.load(f'./models/{name}_blissnet_chkpoint.pth', map_location=self.device)
        self.model.load_state_dict(chkpoint)


    def save(self, name):
        os.makedirs(f"./models", exist_ok=True)

        with open(f"./models/{name}_blissnet_hist.pkl", 'wb') as file:
            data = {'train_loss_s1':self.TRAIN_LOSSES_S1, 'test_loss_s1':self.TEST_LOSSES_S1, 'train_loss_s2':self.TRAIN_LOSSES_S2, 'test_loss_s2':self.TEST_LOSSES_S2}
            pickle.dump(data, file)

        torch.save(self.model.state_dict(), f"./models/{name}_blissnet_chkpoint.pth")

In [7]:
@dataclass
class Configuration():
    batch_size: int
    epochs:int
    lr: float
    emb_dim: int
    K: int
    n_heads: int
    dropout: float
    in_channels: int
    base_channels: int
    n_groups: int
    n_transformer_layers: int
    n_hidden_linear_layers: int
    siren_hidden_dim: int
    siren_layers: int
    omega: int
    grid_size: int

In [8]:
lambda_ep, lambda_coeff, lambda_emb, lambda_gt = 100, 40, 0.01, 0.5

### Training on South Asia Dataset

In [9]:
south_asia_config = Configuration(
    batch_size=32,
    epochs=25,                 
    lr=1e-4,
    emb_dim=512,
    K=64,
    n_heads=8,
    dropout=0.1,
    in_channels=1,
    base_channels=32,
    n_groups=8,
    n_transformer_layers=4,
    n_hidden_linear_layers=3,
    siren_hidden_dim=256,
    siren_layers=4,
    omega=10,
    grid_size=None
)

In [10]:
import gc
gc.collect()

torch.cuda.empty_cache()

In [11]:
south_asia_blissnet = BLISSNetTrainer(south_asia_dataset, south_asia_config, device)

In [12]:
force_train = False
if not force_train and south_asia_blissnet.load('south_asia'):
    print('Loaded checkpoint.')
else:
    south_asia_blissnet.train(lambda_ep, lambda_emb, lambda_coeff, lambda_gt)
    south_asia_blissnet.save('south_asia')
    print('Saved Model!')

========== Training Stage 1 ============


  0%|          | 0/42 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.33 GiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 11.89 GiB is allocated by PyTorch, and 74.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Training on Punjab Dataset

In [ ]:
punjab_config = Configuration(
    batch_size=128,
    epochs=25,                 
    lr=1e-4,
    emb_dim=512,
    K=64,
    n_heads=8,
    dropout=0.1,
    in_channels=1,
    base_channels=32,
    n_groups=8,
    n_transformer_layers=4,
    n_hidden_linear_layers=3,
    siren_hidden_dim=256,
    siren_layers=4,
    omega=10,
    grid_size=None
)

In [ ]:
punjab_blissnet = BLISSNetTrainer(punjab_dataset, punjab_config, device)

In [ ]:
force_train = False
if not force_train and punjab_blissnet.load('punjab'):
    print('Loaded checkpoint.')
else:
    punjab_blissnet.train(lambda_ep, lambda_coeff, lambda_emb, lambda_gt)
    punjab_blissnet.save('punjab')
    print('Saved Model!')

NameError: name 'lambda_ep' is not defined

### Training on Karnatka Dataset

In [ ]:
karnatka_config = Configuration(
    batch_size=128,
    epochs=25,                 
    lr=1e-4,
    emb_dim=512,
    K=64,
    n_heads=8,
    dropout=0.1,
    in_channels=1,
    base_channels=32,
    n_groups=8,
    n_transformer_layers=4,
    n_hidden_linear_layers=3,
    siren_hidden_dim=256,
    siren_layers=4,
    omega=10,
    grid_size=None
)

In [ ]:
karnatka_blissnet = BLISSNetTrainer(karnatka_dataset, karnatka_config, device)

In [ ]:
force_train = False
if not force_train and karnatka_blissnet.load('karnatka'):
    print('Loaded checkpoint.')
else:
    karnatka_blissnet.train(lambda_ep, lambda_coeff, lambda_emb, lambda_gt)
    karnatka_blissnet.save('karnatka')
    print('Saved Model!')